In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from datasets import Dataset, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

project_path = "/content/drive/MyDrive/SPAM_SLM_project"
results_path = f"{project_path}/results"
os.makedirs(results_path, exist_ok=True)

df = pd.read_csv(f"{project_path}/spam_ham_combined.csv")

print("Dataset shape:", df.shape)
print(df['label'].value_counts())

dataset = Dataset.from_pandas(df)

class_labels = ClassLabel(names=["ham", "spam"])
dataset = dataset.cast_column("label", class_labels)

dataset = dataset.train_test_split(
    test_size=0.30,
    stratify_by_column="label",
    seed=42
)

temp_ds = dataset["test"].train_test_split(
    test_size=0.5,
    stratify_by_column="label",
    seed=42
)

train_ds = dataset["train"]
val_ds   = temp_ds["train"]
test_ds  = temp_ds["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

def tokenize_dataset(dataset, tokenizer):
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=256)
    return dataset.map(tokenize, batched=True)

def train_transformer(model_name):
    print(f"\n===== Training {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_tok = tokenize_dataset(train_ds, tokenizer)
    val_tok   = tokenize_dataset(val_ds, tokenizer)
    test_tok  = tokenize_dataset(test_ds, tokenizer)

    data_collator = DataCollatorWithPadding(tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    out_dir = f"{results_path}/{model_name.replace('/', '_')}"
    os.makedirs(out_dir, exist_ok=True)

    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator
    )

    trainer.train()

    preds_output = trainer.predict(test_tok)
    preds = np.argmax(preds_output.predictions, axis=1)
    labels = preds_output.label_ids

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

    pd.DataFrame([metrics]).to_csv(f"{out_dir}/metrics.csv", index=False)
    return metrics

results = {}

for model_name in [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "albert-base-v2"
]:
    results[model_name] = train_transformer(model_name)

print("\n===== Training all-MiniLM-L6-v2 =====")

mini_name = "sentence-transformers/all-MiniLM-L6-v2"
mini_dir = f"{results_path}/all-MiniLM-L6-v2"
os.makedirs(mini_dir, exist_ok=True)

mini_model = SentenceTransformer(mini_name)

X_train = mini_model.encode(train_ds["text"], show_progress_bar=True)
X_test  = mini_model.encode(test_ds["text"], show_progress_bar=True)

y_train = train_ds["label"]
y_test  = test_ds["label"]

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

mini_metrics = {
    "accuracy": accuracy_score(y_test, preds),
    "precision": precision_score(y_test, preds, average="weighted"),
    "recall": recall_score(y_test, preds, average="weighted"),
    "f1": f1_score(y_test, preds, average="weighted")
}

pd.DataFrame([mini_metrics]).to_csv(f"{mini_dir}/metrics.csv", index=False)
results["all-MiniLM-L6-v2"] = mini_metrics

results_df = pd.DataFrame(results).T
results_df.to_csv(f"{project_path}/model_comparison_results.csv")

best_model = results_df["f1"].idxmax()

print("\n===== Final Model Comparison =====")
print(results_df)
print("\nBest model based on F1-score:", best_model)
print("\nAll results saved successfully ✔")

Mounted at /content/drive
Dataset shape: (34518, 2)
label
0    17259
1    17259
Name: count, dtype: int64


Casting the dataset:   0%|          | 0/34518 [00:00<?, ? examples/s]

Train: 24162 | Val: 5178 | Test: 5178

===== Training distilbert-base-uncased =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/756 [00:00<?, ?it/s]

Batches:   0%|          | 0/162 [00:00<?, ?it/s]


===== Final Model Comparison =====
                         accuracy  precision    recall        f1
distilbert-base-uncased  0.990730   0.990731  0.990730  0.990730
bert-base-uncased        0.992275   0.992304  0.992275  0.992275
albert-base-v2           0.990151   0.990159  0.990151  0.990151
all-MiniLM-L6-v2         0.952684   0.952767  0.952684  0.952682

Best model based on F1-score: bert-base-uncased

All results saved successfully ✔
